In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

!pip install -q --upgrade ultralytics

import os
import yaml
from ultralytics import YOLO

data_path = '/kaggle/input/datasets/ramgolla17042006/cplid-updated'
classes = ["insulator", "defect"]
nc = len(classes)

data_yaml_dict = {
    'path': data_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test' if os.path.exists(os.path.join(data_path, 'images/test')) else 'images/val',
    'nc': nc,
    'names': classes
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml_dict, f, default_flow_style=False)

print(" data.yaml created")

custom_yaml = f"""
nc: {nc}
scales:
  n: [0.33, 0.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C3k2, [128, True, 0.25]]
  - [-1, 1, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C3k2, [256, True, 0.25]]
  - [-1, 1, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C3k2, [512, True, 0.5]]
  - [-1, 1, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C3k2, [1024, True]]
  - [-1, 1, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C3k2, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C3k2, [1024, False]]
  - [[19, 22, 25], 1, Detect, [nc]]
"""

with open('insdef_c2f_yolo.yaml', 'w') as f:
    f.write(custom_yaml)

print(" YAML saved (C2f version)")

print("\nBuilding YOLOv11 + C2f Backbone...")
model = YOLO('insdef_c2f_yolo.yaml')
model = model.load('yolo11n.pt')

model.info()

print("\nStarting Training...")
results = model.train(
    data='data.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    patience=40,
    optimizer='auto',
    plots=True,
    save=True,
    name='insdef_c2f'
)

print("\nEvaluating on test set...")
test_results = model.val(data='data.yaml', split='test', batch=4, plots=True)

print("\n" + "="*60)
print("RESULTS - YOLOv11 + C2f Version")
print("="*60)
print(f"mAP@0.5       : {test_results.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {test_results.box.map:.4f}")
print(f"Precision     : {test_results.box.p.mean():.4f}")
print(f"Recall        : {test_results.box.r.mean():.4f}")
print("="*60)